# 01 — Sanity checks

This notebook is the correctness gate for the Yamada benchmarks. It first runs the published/independent algebraic checks in `dev/run_yamada_sanity_checks.py`, then performs a broader projection-invariance test on **25 deterministic trivalent spatial graphs**.

The projection test intentionally mixes five abstract trivalent families — theta, \(K_4\), triangular prism, \(K_{3,3}\), and cube — with five deterministic spatial embeddings of each family. For every fixed spatial embedding, different valid generic projections may have different crossing counts and different PD codes, but the **normalized Yamada polynomial must remain unchanged**.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

script = ROOT / "dev" / "run_yamada_sanity_checks.py"
proc = subprocess.run(
    [sys.executable, str(script)],
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError(
        f"Sanity checks failed.\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
    )
assert "PASS: all published/independent Yamada sanity checks succeeded." in proc.stdout
print("Branch-local import:", kg_path)


## Projection invariance on 25 deterministic trivalent spatial graphs

For each case,
\[
G_{3D}\longrightarrow \text{many generic projections}\longrightarrow
\operatorname{PD}(G)\longrightarrow \Upsilon(G;A).
\]

The embedding is held fixed while the projection direction changes. Every graph is checked to be connected and exactly trivalent before use.

The 25 cases are:

- 5 theta embeddings (\(V=2,E=3\));
- 5 \(K_4\) embeddings (\(V=4,E=6\));
- 5 triangular-prism embeddings (\(V=6,E=9\));
- 5 \(K_{3,3}\) embeddings (\(V=6,E=9\));
- 5 cube embeddings (\(V=8,E=12\)).

The five variants within each family are deterministic 3-D embeddings generated from different fixed seeds. They are not counted as different abstract graphs; the purpose is to test the full projection/PD/Yamada pathway across a wider range of trivalent geometry and combinatorics.


In [ ]:
import networkx as nx
import numpy as np
import sympy as sp

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
)

A = sp.Symbol("A")
PROJECTION_SAMPLES = 100
MIN_VALID_PROJECTIONS = 5


def _same_polynomial(left, right):
    return sp.simplify(sp.together(sp.expand(left - right))) == 0


def _theta_embedding(variant):
    """Five deterministic theta embeddings, including crossing/mirror variants."""
    zsign = -1.0 if variant in {2, 4} else 1.0
    amp = [0.75, 1.00, 1.15, 0.90, 1.25][variant]
    zamp = [0.25, 0.50, 0.65, 0.40, 0.55][variant]
    upper = [1.55, 2.00, 2.25, 1.80, 2.40][variant]

    graph = nx.MultiGraph()
    graph.add_node("u", pos=np.array([-2.0, 0.0, 0.0]))
    graph.add_node("v", pos=np.array([ 2.0, 0.0, 0.0]))
    curves = [
        np.array(
            [[-2, 0, 0], [-1, -amp, zamp*zsign],
             [1, amp, zamp*zsign], [2, 0, 0]],
            dtype=float,
        ),
        np.array(
            [[-2, 0, 0], [-1, amp, -zamp*zsign],
             [1, -amp, -zamp*zsign], [2, 0, 0]],
            dtype=float,
        ),
        np.array(
            [[-2, 0, 0], [-1, upper, 0.12*variant],
             [1, upper, -0.12*variant], [2, 0, 0]],
            dtype=float,
        ),
    ]

    # A fixed orientation-preserving affine perturbation makes the five
    # cases geometrically distinct without changing the embedded graph type.
    angle = 0.17 * variant
    c, s = np.cos(angle), np.sin(angle)
    M = np.array(
        [[c, -s, 0.05*variant],
         [s,  c, 0.03*(variant-2)],
         [0.0, 0.0, 1.0]],
        dtype=float,
    )
    for pts in curves:
        pts[:] = pts @ M.T
    left, right = curves[0][0].copy(), curves[0][-1].copy()
    graph.nodes["u"]["pos"] = left
    graph.nodes["v"]["pos"] = right
    for pts in curves:
        pts[0] = left
        pts[-1] = right
        graph.add_edge("u", "v", pts=pts)
    return graph


def _simple_trivalent_embedding(abstract_graph, seed):
    """Deterministic generic 3-D straight-edge embedding of a simple cubic graph."""
    abstract_graph = nx.Graph(abstract_graph)
    if not nx.is_connected(abstract_graph):
        raise ValueError("benchmark abstract graph must be connected")
    if not all(degree == 3 for _, degree in abstract_graph.degree()):
        raise ValueError("benchmark abstract graph must be exactly trivalent")

    positions = nx.spring_layout(
        abstract_graph,
        dim=3,
        seed=seed,
        iterations=500,
        scale=2.0,
    )
    nodes = sorted(abstract_graph.nodes(), key=repr)
    xyz = np.array([positions[node] for node in nodes], dtype=float)

    # Apply a deterministic nonsingular affine perturbation. This avoids
    # special-coordinate degeneracies while retaining the same spatial embedding.
    rng = np.random.default_rng(seed + 100_000)
    shear = np.eye(3) + rng.uniform(-0.08, 0.08, size=(3, 3))
    if np.linalg.det(shear) < 0:
        shear[0] *= -1
    xyz = xyz @ shear.T
    xyz -= xyz.mean(axis=0)

    P = {node: xyz[i] for i, node in enumerate(nodes)}
    graph = nx.MultiGraph()
    for node in nodes:
        graph.add_node(node, pos=P[node].copy())
    for u, v in abstract_graph.edges():
        graph.add_edge(
            u,
            v,
            pts=np.linspace(P[u], P[v], 5),
        )
    return graph


def _make_25_cases():
    families = [
        ("theta", None),
        ("K4", nx.complete_graph(4)),
        ("triangular_prism", nx.circular_ladder_graph(3)),
        ("K3_3", nx.complete_bipartite_graph(3, 3)),
        ("cube", nx.cubical_graph()),
    ]

    cases = []
    for family, abstract in families:
        for variant in range(5):
            if family == "theta":
                graph = _theta_embedding(variant)
            else:
                graph = _simple_trivalent_embedding(
                    abstract,
                    seed=20260819 + 1000*families.index((family, abstract)) + variant,
                )
            cases.append((f"{family}_{variant+1}", graph))

    assert len(cases) == 25
    return cases


TRIVALENT_CASES = _make_25_cases()

for label, graph in TRIVALENT_CASES:
    degrees = sorted(dict(graph.degree()).values())
    assert nx.is_connected(nx.Graph(graph)), f"{label}: graph is disconnected"
    assert degrees and all(d == 3 for d in degrees), (
        f"{label}: graph is not exactly trivalent: {degrees}"
    )

print("Prepared 25 deterministic connected trivalent spatial graphs.")
for label, graph in TRIVALENT_CASES:
    print(
        f"{label:24s} V={graph.number_of_nodes():2d} "
        f"E={graph.number_of_edges():2d}"
    )


In [ ]:
def _check_projection_invariance(label, graph):
    degrees = sorted(dict(graph.degree()).values())
    assert degrees and all(d == 3 for d in degrees), (
        f"{label}: benchmark graph is not trivalent: {degrees}"
    )

    projections = sample_projections(
        graph,
        num_rotation_samples=PROJECTION_SAMPLES,
    )
    if len(projections) < MIN_VALID_PROJECTIONS:
        raise AssertionError(
            f"{label}: only {len(projections)}/{PROJECTION_SAMPLES} sampled views were valid; "
            f"expected at least {MIN_VALID_PROJECTIONS}."
        )

    records = []
    for projection in projections:
        result = compute_yamada_polynomial(
            graph,
            A,
            rotation_angles=projection.rotation_angles,
            rotation_order=projection.rotation_order,
            crossing_warning_threshold=None,
            normalize=True,
            n_jobs=1,
            method="negami",
            return_result=True,
        )
        assert result.projection.pd_code == projection.pd_code
        assert result.projection.num_crossings == projection.num_crossings
        records.append((projection, sp.expand(result.polynomial)))

    reference = records[0][1]
    mismatches = [
        (index, projection.rotation_angles, projection.num_crossings, polynomial)
        for index, (projection, polynomial) in enumerate(records)
        if not _same_polynomial(polynomial, reference)
    ]
    if mismatches:
        details = "\n".join(
            f"  sample={index}, angles={angles}, crossings={crossings}, Yamada={polynomial}"
            for index, angles, crossings, polynomial in mismatches[:5]
        )
        raise AssertionError(
            f"{label}: normalized Yamada changed across projections.\n"
            f"reference={reference}\n{details}"
        )

    crossing_counts = [projection.num_crossings for projection, _ in records]
    distinct_pd_codes = len({projection.pd_code for projection, _ in records})
    print(
        f"PASS  {label:24s}: {len(records)}/{PROJECTION_SAMPLES} valid projections, "
        f"crossings={min(crossing_counts)}..{max(crossing_counts)}, "
        f"distinct PD codes={distinct_pd_codes}, distinct Yamada values=1"
    )
    return records


projection_invariance_records = {}
for label, graph in TRIVALENT_CASES:
    projection_invariance_records[label] = _check_projection_invariance(label, graph)

assert len(projection_invariance_records) == 25
print(
    "PASS: normalized Yamada is projection-invariant across all valid sampled "
    "projections of all 25 deterministic trivalent benchmark embeddings."
)


### Interpretation

A pass means that the complete
\[
G_{3D}\to\text{rotation/projection}\to\text{crossings/arcs}\to
\text{PD code}\to\Upsilon
\]
pipeline produced exactly one normalized Yamada value for each fixed spatial embedding, even when different projection directions produced different diagrams.

This is a projection-invariance sanity check, not a proof that the 25 spatial embeddings are pairwise inequivalent. Notebook 05 addresses the separate question of independently certified distinct embeddings of the same abstract graph.
